In [1]:
%pip install transformers datasets accelerate

     ---------------------------------------- 0.0/41.5 kB ? eta -:--:--
     ---------------------------------------- 41.5/41.5 kB ? eta 0:00:00
     ---------------------------------------- 0.0/57.7 kB ? eta -:--:--
     ---------------------------------------- 57.7/57.7 kB ? eta 0:00:00
  Using cached requests-2.32.3-py3-none-any.whl.metadata (4.6 kB)
     ---------------------------------------- 0.0/74.8 kB ? eta -:--:--
     ---------------------------------------- 74.8/74.8 kB 4.0 MB/s eta 0:00:00
   ---------------------------------------- 0.0/10.4 MB ? eta -:--:--
   -- ------------------------------------- 0.6/10.4 MB 19.5 MB/s eta 0:00:01
   ------- -------------------------------- 1.9/10.4 MB 29.7 MB/s eta 0:00:01
   --------------------- ------------------ 5.5/10.4 MB 44.0 MB/s eta 0:00:01
   ----------------------------- ---------- 7.8/10.4 MB 45.1 MB/s eta 0:00:01
   ---------------------------------------  10.4/10.4 MB 46.9 MB/s eta 0:00:01
   ----------------------------


[notice] A new release of pip is available: 24.0 -> 25.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer

model_name = "gpt2"
model = GPT2LMHeadModel.from_pretrained(model_name)
tokenizer = GPT2Tokenizer.from_pretrained(model_name)

# Importante para evitar warnings con tokens especiales
tokenizer.pad_token = tokenizer.eos_token
model.resize_token_embeddings(len(tokenizer))

c:\Users\tinch\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\tinch\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:144: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\tinch\.cache\huggingface\hub\models--gpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In or

Embedding(50257, 768)

In [3]:
# Prueba del modelo
article_text = """El presidente dio una conferencia sobre política internacional, abordando temas clave como comercio exterior, relaciones diplomáticas con Asia y la crisis humanitaria en Europa del Este."""

# Formar el prompt como lo entrenaste:
prompt = f"Article: {article_text.strip()}\n\nSummary:"

# Tokenizar
inputs = tokenizer(prompt, return_tensors="pt")

# Generar texto (usa CPU o GPU si está disponible)
output = model.generate(
    **inputs,
    max_new_tokens=100,
    do_sample=True,
    temperature=0.7,
    top_k=50,
    top_p=0.95,
    num_return_sequences=1
)

# Decodificar
generated_text = tokenizer.decode(output[0], skip_special_tokens=True)
print(generated_text)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Article: El presidente dio una conferencia sobre política internacional, abordando temas clave como comercio exterior, relaciones diplomáticas con Asia y la crisis humanitaria en Europa del Este.

Summary: On 28 December 2015, the Russian Federation decided to provide humanitarian aid to the people of Central Asia in response to the worsening humanitarian situation in the country. The mission of the Russian Federation was to provide humanitarian assistance to the people of Central Asia to ensure the safety, health and well-being of the people of Central Asia.

The mission is also to provide humanitarian assistance to the people of Central Asia and to the international community to protect them from the economic and political instability of the region, and to


In [4]:
from datasets import load_dataset

# Cargar el dataset
dataset = load_dataset("cnn_dailymail", "3.0.0", split="train")

# Concatenar artículo y resumen como texto
def format_for_causal_lm(example):
    return {
        "text": f"Article: {example['article']}\n\nSummary: {example['highlights']}"
    }

formatted_dataset = dataset.map(format_for_causal_lm)

c:\Users\tinch\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:144: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\tinch\.cache\huggingface\hub\datasets--cnn_dailymail. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Map: 100%|██████████| 287113/287113 [00:17<00:00, 16127.34 examples/s]


In [5]:
dataset

Dataset({
    features: ['article', 'highlights', 'id'],
    num_rows: 287113
})

In [6]:
formatted_dataset

Dataset({
    features: ['article', 'highlights', 'id', 'text'],
    num_rows: 287113
})

In [7]:
def filter_long_text(example):
  return len(example["text"].split(" ")) < 512

filtered_dataset = formatted_dataset.filter()

Filter: 100%|██████████| 287113/287113 [00:03<00:00, 74160.25 examples/s]


In [8]:
def tokenize_function(example):
    tokens = tokenizer(example["text"], truncation=True, padding="max_length", max_length=512)
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

tokenized_datasets = formatted_dataset.map(tokenize_function, batched=True, remove_columns=["text"])

Map: 100%|██████████| 287113/287113 [21:33<00:00, 221.94 examples/s]


In [20]:
# 1% del dataset
k = round(len(tokenized_datasets)*0.01)
print(k)

2871


In [21]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="./gpt2-finetuned",  # carpeta temporal de salida
    per_device_train_batch_size=2,
    num_train_epochs=3,
    save_strategy="epoch",          # solo guarda al final de cada época
    save_total_limit=1,             # solo guarda el último
    logging_steps=100,
    report_to="none",
    #fp16=True,                      # opcional: solo si tu GPU lo soporta
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets.select(range(k)),
    tokenizer=tokenizer,
)

C:\Users\tinch\AppData\Local\Temp\ipykernel_21896\3817378536.py:14: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [22]:
trainer.train()

Step,Training Loss
100,2.872400
200,2.859400
300,2.828800
400,2.738500
500,2.748600
600,2.821200
700,2.758800
800,2.830200
900,2.806700
1000,2.778100


c:\Users\tinch\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
c:\Users\tinch\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


TrainOutput(global_step=4308, training_loss=2.6031374532864286, metrics={'train_runtime': 31640.301, 'train_samples_per_second': 0.272, 'train_steps_per_second': 0.136, 'total_flos': 2250508271616000.0, 'train_loss': 2.6031374532864286, 'epoch': 3.0})

In [ ]:
from huggingface_hub import login, HfApi, HfHubHTTPError
import os
#from google.colab import userdata

HF_TOKEN = "" #userdata.get('HF_TOKEN')
login(token=HF_TOKEN)
repo_id = "Pseudokiwi/gpt2-finetuned"

In [ ]:
api = HfApi()
try:
    api.create_repo(repo_id=repo_id, private=True)
except HfHubHTTPError as e:
    print("Repo ya existe o no se puede crear:", e)

In [ ]:
api.update_repo_settings(repo_id="Pseudokiwi/gpt2-finetuned", private=True)

In [34]:
trainer.push_to_hub()

No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/Pseudokiwi/gpt2-finetuned/commit/ab504a0633ae711772d9c6dc0fa647a0c1beafa1', commit_message='End of training', commit_description='', oid='ab504a0633ae711772d9c6dc0fa647a0c1beafa1', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Pseudokiwi/gpt2-finetuned', endpoint='https://huggingface.co', repo_type='model', repo_id='Pseudokiwi/gpt2-finetuned'), pr_revision=None, pr_num=None)